In [ ]:
import rasterio
import pandas as pd
import numpy as np
import os
from datetime import datetime

def find_nearest_pixel(dataset, target_lat, target_lon):
    """
    Find the pixel indices in a TIFF file closest to the target latitude and longitude.

    Args:
        dataset (rasterio.DatasetReader): Opened TIFF dataset.
        target_lat (float): Target latitude.
        target_lon (float): Target longitude.

    Returns:
        tuple: (row, col) indices of the nearest pixel.
    """
    # Get the transform (maps pixel coordinates to geographic coordinates)
    transform = dataset.transform
    # Convert lat/lon to pixel coordinates
    col, row = ~transform * (target_lon, target_lat)
    row, col = int(round(row)), int(round(col))
    # Ensure indices are within bounds
    row = max(0, min(row, dataset.height - 1))
    col = max(0, min(col, dataset.width - 1))
    return row, col

def process_chirps_tiff_to_csv(cities, tiff_directory, output_csv_path):
    """
    Process CHIRPS v3 monthly TIFF files to extract rainfall_mm and derive
    rainfall_monthly_anomaly, rainfall_3month_avg, and rainfall_6month_avg for each city.

    Args:
        cities (dict): Dictionary of city names and (lat, lon) tuples.
        tiff_directory (str): Directory containing CHIRPS TIFF files (format: chirps-v3.0.YYYY.MM.tif).
        output_csv_path (str): Path to save the output CSV file.
    """
    all_city_dfs = []

    # --- 1. Collect all TIFF files and sort by date ---
    tiff_files = [f for f in os.listdir(tiff_directory) if f.endswith('.tif') and 'chirps-v3.0' in f]
    # Extract year and month from filenames (assuming format chirps-v3.0.YYYY.MM.tif)
    tiff_files_info = []
    for f in tiff_files:
        try:
            parts = f.split('.')
            year, month = int(parts[2]), int(parts[3])
            tiff_files_info.append((f, year, month))
        except (IndexError, ValueError):
            print(f"WARNING: Skipping file with unexpected name format: {f}")
            continue
    # Sort by year and month
    tiff_files_info.sort(key=lambda x: (x[1], x[2]))

    # --- 2. Process each city ---
    for city_name, (lat, lon) in cities.items():
        print(f"\n--- Processing data for {city_name} ---")
        city_data = {'city': [], 'date': [], 'rainfall_mm': []}

        # Extract rainfall_mm for each TIFF file
        for tiff_file, year, month in tiff_files_info:
            tiff_path = os.path.join(tiff_directory, tiff_file)
            try:
                with rasterio.open(tiff_path) as dataset:
                    # Find nearest pixel
                    row, col = find_nearest_pixel(dataset, lat, lon)
                    # Read rainfall data (assuming single band)
                    rainfall = dataset.read(1)[row, col]
                    # Check for invalid/missing data (CHIRPS uses -9999 for missing)
                    if rainfall == -9999 or np.isnan(rainfall):
                        print(f"    WARNING: Invalid rainfall value at {year}-{month:02d} for {city_name}")
                        rainfall = np.nan
                    else:
                        print(f"    Extracted rainfall_mm = {rainfall:.2f} for {city_name} at {year}-{month:02d}")
                    # Store data
                    city_data['city'].append(city_name)
                    city_data['date'].append(f"{year}-{month:02d}")
                    city_data['rainfall_mm'].append(rainfall)
            except Exception as e:
                print(f"    ERROR: Failed to process {tiff_file} for {city_name}. Error: {e}")
                city_data['city'].append(city_name)
                city_data['date'].append(f"{year}-{month:02d}")
                city_data['rainfall_mm'].append(np.nan)

        # Create DataFrame for the city
        city_df = pd.DataFrame(city_data)
        city_df['date'] = pd.to_datetime(city_df['date'], format='%Y-%m')

        # --- 3. Compute long-term monthly averages ---
        # Extract month number for grouping
        city_df['month'] = city_df['date'].dt.month
        # Compute mean rainfall for each month (January–December)
        monthly_means = city_df.groupby('month')['rainfall_mm'].mean().reindex(range(1, 13))
        # Map month number to monthly mean
        city_df['rainfall_monthly_anomaly'] = city_df.apply(
            lambda row: row['rainfall_mm'] - monthly_means[row['month']] if not np.isnan(row['rainfall_mm']) else np.nan,
            axis=1
        )

        # --- 4. Compute rolling averages ---
        # 3-month rolling average
        city_df['rainfall_3month_avg'] = city_df['rainfall_mm'].rolling(window=3, min_periods=3).mean()
        # 6-month rolling average
        city_df['rainfall_6month_avg'] = city_df['rainfall_mm'].rolling(window=6, min_periods=6).mean()

        # Drop temporary month column
        city_df = city_df.drop(columns=['month'])

        all_city_dfs.append(city_df)

    # --- 5. Combine and save to CSV ---
    if not all_city_dfs:
        print("No data was processed. Exiting.")
        return

    print("\n--- Combining data for all cities and saving to CSV ---")
    final_df = pd.concat(all_city_dfs, ignore_index=True)
    # Sort by city and date for readability
    final_df = final_df.sort_values(['city', 'date'])
    # Format date as YYYY-MM for CSV output
    final_df['date'] = final_df['date'].dt.strftime('%Y-%m')

    final_df.to_csv(output_csv_path, index=False)
    print(f"\nSuccessfully created CSV file at: {output_csv_path}")
    print(f"Final dataset has {final_df.shape[0]} rows and {final_df.shape[1]} columns.")
    # Check for columns with all NaN values
    nan_columns = final_df.columns[final_df.isna().all()].tolist()
    if nan_columns:
        print(f"WARNING: The following columns contain only NaN values: {nan_columns}")

# --- Main Execution Block ---
if __name__ == '__main__':
    CITIES_TO_PROCESS = {
        'Kampala': (0.3152, 32.5816),
        'Kabale': (-1.2420, 29.9856),
        'Kasese': (0.1699, 30.0781)
    }
    TIFF_DIR = 'tifs'  # Update this to your TIFF files directory
    OUTPUT_CSV = 'chirps_city_rainfall_metrics.csv'
    process_chirps_tiff_to_csv(CITIES_TO_PROCESS, TIFF_DIR, OUTPUT_CSV)

In [ ]:
import netCDF4
import pandas as pd
import numpy as np
import os

def find_nearest_grid_indices(lats, lons, target_lat, target_lon):
    """
    Finds the indices of the grid cell closest to a target latitude and longitude.

    Args:
        lats (np.array): Array of latitudes from the NetCDF grid.
        lons (np.array): Array of longitudes from the NetCDF grid.
        target_lat (float): Latitude of the target location (e.g., a city).
        target_lon (float): Longitude of the target location.

    Returns:
        tuple: A tuple containing the index for latitude and the index for longitude.
    """
    dist_sq = (lats - target_lat)**2 + (lons - target_lon)**2
    min_dist_idx = dist_sq.argmin()
    lat_idx, lon_idx = np.unravel_index(min_dist_idx, lats.shape)
    return lat_idx, lon_idx

def process_all_data_to_csv(cities, data_directory, output_csv_path):
    """
    Processes multiple NetCDF files (one time-series, several static) for a list of cities,
    combines them, and saves the result to a single CSV file with meaningful column names.

    Args:
        cities (dict): A dictionary where keys are city names and values are (lat, lon) tuples.
        data_directory (str): The path to the directory containing all the .nc files.
        output_csv_path (str): The path where the final combined CSV will be saved.
    """
    # Define a mapping of CDS variable names to meaningful names
    variable_name_mapping = {
        't2m': 'temperature_2m',
        'd2m': 'dewpoint_temperature_2m',
        'tp': 'total_precipitation',
        'sp': 'surface_pressure',
        'u10': 'u_component_wind_10m',
        'v10': 'v_component_wind_10m',
        'skt': 'skin_temperature',
        'stl1': 'soil_temperature_level1',
        'stl2': 'soil_temperature_level2',
        'stl3': 'soil_temperature_level3',
        'stl4': 'soil_temperature_level4',
        'src': 'skin_reservoir_content',
        'swvl1': 'soil_volume_water_content_level1',
        'swvl2': 'soil_volume_water_content_level2',
        'swvl3': 'soil_volume_water_content_level3',
        'swvl4': 'soil_volume_water_content_level4',
        'ro': 'runoff',
        'e': 'evaporation',
        'cl': 'lake_cover',
        'cvl': 'low_vegetation_cover',
        'cvh': 'high_vegetation_cover',
        'z': 'geopotential_height',
        'lsm': 'land_sea_mask',
        'slt': 'soil_type',
        'tvl': 'low_vegetation_type',
        'tvh': 'high_vegetation_type',
        'expver': 'experiment_version'
    }

    # Define static file variable names and their corresponding files
    static_files_info = {
        'cl': 'clake.area-subset.5.35.-1.29.nc',
        'cvl': 'cvl.area-subset.5.35.-1.29.nc',
        'cvh': 'cvh.area-subset.5.35.-1.29.nc',
        'z': 'geo.area-subset.5.35.-1.29.nc',
        'lsm': 'lsm.area-subset.5.35.-1.29.nc',
        'slt': 'slt.area-subset.5.35.-1.29.nc',
        'tvl': 'tvl.area-subset.5.35.-1.29.nc',
        'tvh': 'tvh.area-subset.5.35.-1.29.nc',
    }

    all_city_dfs = []

    # --- 1. Open Main Time-Series File and Get Grid Info ---
    main_ts_file = os.path.join(data_directory, 'data_stream-moda.nc')
    print(f"Reading main time-series file: {main_ts_file}")
    if not os.path.exists(main_ts_file):
        print(f"FATAL ERROR: Main file not found at {main_ts_file}")
        return
        
    with netCDF4.Dataset(main_ts_file, 'r') as nc_main:
        lat_vals = nc_main.variables['latitude'][:]
        lon_vals = nc_main.variables['longitude'][:]
        time_vals = nc_main.variables['valid_time'][:]
        
        lon_grid, lat_grid = np.meshgrid(lon_vals, lat_vals)

        # --- 2. Loop Through Each City ---
        for city_name, (lat, lon) in cities.items():
            print(f"\n--- Processing data for {city_name} ---")
            lat_idx, lon_idx = find_nearest_grid_indices(lat_grid, lon_grid, lat, lon)
            print(f"Found nearest grid point for {city_name} at (lat_idx={lat_idx}, lon_idx={lon_idx})")
            print(f"  Target Coords: ({lat:.4f}, {lon:.4f})")
            print(f"  Grid Coords:   ({lat_vals[lat_idx]:.4f}, {lon_vals[lon_idx]:.4f})")

            # --- 3. Extract Time-Series Data for the City ---
            city_data = {'city': city_name}
            datetimes = pd.to_datetime(time_vals, unit='s', origin='1970-01-01')
            city_data['datetime'] = datetimes
            city_data['target_latitude'] = lat
            city_data['target_longitude'] = lon
            city_data['grid_latitude'] = lat_vals[lat_idx]
            city_data['grid_longitude'] = lon_vals[lon_idx]

            for var_name, variable in nc_main.variables.items():
                if variable.dimensions == ('valid_time', 'latitude', 'longitude'):
                    print(f"  Extracting time-series for: {var_name}")
                    city_data[var_name] = variable[:, lat_idx, lon_idx]
            
            if 'expver' in nc_main.variables:
                city_data['expver'] = nc_main.variables['expver'][:]

            city_df = pd.DataFrame(city_data)

            # --- 4. Extract and Add Static Data for the City ---
            for static_var_name, static_filename in static_files_info.items():
                static_filepath = os.path.join(data_directory, static_filename)
                print(f"  Extracting static data from: {static_filename}")
                if not os.path.exists(static_filepath):
                    print(f"    WARNING: Static file not found, skipping: {static_filepath}")
                    city_df[static_var_name] = np.nan
                    continue

                with netCDF4.Dataset(static_filepath, 'r') as nc_static:
                    # Print all variables and dimensions for debugging
                    print(f"    Variables in {static_filename}: {list(nc_static.variables.keys())}")
                    
                    if static_var_name in nc_static.variables:
                        v_obj = nc_static.variables[static_var_name]
                        print(f"    Dimensions of {static_var_name}: {v_obj.dimensions}")
                        print(f"    Shape of {static_var_name}: {v_obj.shape}")
                        
                        # Handle (time, latitude, longitude) by selecting the first time step
                        if v_obj.dimensions[:3] == ('time', 'latitude', 'longitude'):
                            try:
                                # Check time dimension length
                                time_len = v_obj.shape[0]
                                print(f"    Time dimension length for {static_var_name}: {time_len}")
                                static_value = v_obj[0, lat_idx, lon_idx]  # Select first time step
                                # Check if value is masked or invalid
                                if np.ma.is_masked(static_value) or np.isnan(static_value):
                                    print(f"    WARNING: {static_var_name} value is invalid (NaN or masked) for {city_name}")
                                    city_df[static_var_name] = np.nan
                                else:
                                    print(f"    Extracted {static_var_name} = {static_value} for {city_name}")
                                    city_df[static_var_name] = static_value
                            except Exception as e:
                                print(f"    ERROR: Failed to extract {static_var_name} from {static_filename}. Error: {e}")
                                city_df[static_var_name] = np.nan
                        elif v_obj.dimensions == ('latitude', 'longitude'):
                            try:
                                static_value = v_obj[lat_idx, lon_idx]
                                # Check if value is masked or invalid
                                if np.ma.is_masked(static_value) or np.isnan(static_value):
                                    print(f"    WARNING: {static_var_name} value is invalid (NaN or masked) for {city_name}")
                                    city_df[static_var_name] = np.nan
                                else:
                                    print(f"    Extracted {static_var_name} = {static_value} for {city_name}")
                                    city_df[static_var_name] = static_value
                            except Exception as e:
                                print(f"    ERROR: Failed to extract {static_var_name} from {static_filename}. Error: {e}")
                                city_df[static_var_name] = np.nan
                        else:
                            print(f"    WARNING: Unexpected dimensions for {static_var_name} in {static_filename}: {v_obj.dimensions}")
                            city_df[static_var_name] = np.nan
                    else:
                        print(f"    WARNING: Variable {static_var_name} not found in {static_filename}")
                        city_df[static_var_name] = np.nan

            all_city_dfs.append(city_df)

    # --- 5. Combine DataFrames and Save to CSV ---
    if not all_city_dfs:
        print("No data was processed. Exiting.")
        return
        
    print("\n--- Combining data for all cities and saving to CSV ---")
    final_df = pd.concat(all_city_dfs, ignore_index=True)

    # Rename columns using the variable_name_mapping
    final_df = final_df.rename(columns=variable_name_mapping)

    # Reorder columns for better readability
    id_cols = ['city', 'datetime', 'target_latitude', 'target_longitude', 'grid_latitude', 'grid_longitude']
    data_cols = [col for col in final_df.columns if col not in id_cols]
    final_df = final_df[id_cols + sorted(data_cols)]

    final_df.to_csv(output_csv_path, index=False)
    print(f"\nSuccessfully created CSV file at: {output_csv_path}")
    print(f"Final dataset has {final_df.shape[0]} rows and {final_df.shape[1]} columns.")
    # Print columns with all NaN values for debugging
    nan_columns = final_df.columns[final_df.isna().all()].tolist()
    if nan_columns:
        print(f"WARNING: The following columns contain only NaN values: {nan_columns}")

# --- Main Execution Block ---
if __name__ == '__main__':
    CITIES_TO_PROCESS = {
        'Kampala': (0.3152, 32.5816),
        'Kabale': (-1.2420, 29.9856),
        'Kasese': (0.1699, 30.0781)
    }
    DATA_DIR = 'cds data/netcdf_data/full_data'
    OUTPUT_CSV = 'combined_city_weather_and_static_data.csv'
    process_all_data_to_csv(CITIES_TO_PROCESS, DATA_DIR, OUTPUT_CSV)

### Joining the CSV files

In [ ]:
import pandas as pd
import numpy as np

def merge_chirps_era5_csv(chirps_csv_path, era5_csv_path, output_csv_path):
    """
    Merge CHIRPS rainfall CSV and ERA5-Land weather/static CSV into a single CSV file,
    using ERA5-Land as the base dataset and filling missing CHIRPS data with city-specific means.

    Args:
        chirps_csv_path (str): Path to the CHIRPS CSV file.
        era5_csv_path (str): Path to the ERA5-Land CSV file.
        output_csv_path (str): Path to save the merged CSV file.
    """
    # --- 1. Read the CSV files ---
    print("Reading CHIRPS CSV file...")
    chirps_df = pd.read_csv(chirps_csv_path)
    print(f"CHIRPS CSV shape: {chirps_df.shape}")

    print("Reading ERA5-Land CSV file...")
    era5_df = pd.read_csv(era5_csv_path)
    print(f"ERA5-Land CSV shape: {era5_df.shape}")

    # --- 2. Prepare ERA5-Land data ---
    # Convert datetime to YYYY-MM format
    era5_df['datetime'] = pd.to_datetime(era5_df['datetime'])
    era5_df['date'] = era5_df['datetime'].dt.strftime('%Y-%m')

    # Define columns to aggregate
    static_cols = [
        'geopotential_height', 'high_vegetation_cover', 'high_vegetation_type',
        'lake_cover', 'land_sea_mask', 'low_vegetation_cover', 'low_vegetation_type',
        'soil_type'
    ]
    cumulative_cols = ['total_precipitation', 'runoff', 'evaporation']
    continuous_cols = [
        col for col in era5_df.columns 
        if col not in static_cols + cumulative_cols + ['city', 'datetime', 'date', 
                                                      'target_latitude', 'target_longitude', 
                                                      'grid_latitude', 'grid_longitude']
    ]

    # Aggregate ERA5-Land data to monthly resolution
    print("Aggregating ERA5-Land data to monthly resolution...")
    agg_dict = {col: 'first' for col in static_cols + ['target_latitude', 'target_longitude', 'grid_latitude', 'grid_longitude']}
    agg_dict.update({col: 'sum' for col in cumulative_cols if col in era5_df.columns})
    agg_dict.update({col: 'mean' for col in continuous_cols if col in era5_df.columns})

    era5_monthly = era5_df.groupby(['city', 'date']).agg(agg_dict).reset_index()
    print(f"ERA5-Land monthly aggregated shape: {era5_monthly.shape}")

    # --- 3. Merge the DataFrames ---
    print("Merging CHIRPS and ERA5-Land data...")
    # Ensure date columns are strings for merging
    chirps_df['date'] = chirps_df['date'].astype(str)
    era5_monthly['date'] = era5_monthly['date'].astype(str)

    # Merge with ERA5-Land as base (left join)
    merged_df = pd.merge(
        era5_monthly,
        chirps_df,
        on=['city', 'date'],
        how='left',
        suffixes=('_era5', '_chirps')
    )
    print(f"Merged DataFrame shape: {merged_df.shape}")

    # --- 4. Fill missing CHIRPS data with city-specific means ---
    chirps_cols = ['rainfall_mm', 'rainfall_monthly_anomaly', 'rainfall_3month_avg', 'rainfall_6month_avg']
    print("Filling missing CHIRPS data with city-specific means...")
    for city in merged_df['city'].unique():
        city_mask = merged_df['city'] == city
        for col in chirps_cols:
            if col in merged_df.columns:
                city_mean = merged_df.loc[city_mask, col].mean()
                if not np.isnan(city_mean):
                    merged_df.loc[city_mask, col] = merged_df.loc[city_mask, col].fillna(city_mean)
                    print(f"  Filled missing {col} for {city} with mean: {city_mean:.2f}")
                else:
                    print(f"  WARNING: Could not compute mean for {col} in {city} (all NaN)")

    # --- 5. Clean up and sort ---
    # Drop redundant columns (e.g., target_latitude_chirps if same as target_latitude_era5)
    for col in ['target_latitude', 'target_longitude', 'grid_latitude', 'grid_longitude']:
        chirps_col = f'{col}_chirps'
        era5_col = f'{col}_era5'
        if chirps_col in merged_df.columns and era5_col in merged_df.columns:
            # Check if values are identical (ignoring NaN)
            if merged_df[chirps_col].dropna().equals(merged_df[era5_col].dropna()):
                merged_df = merged_df.drop(columns=chirps_col)
                merged_df = merged_df.rename(columns={era5_col: col})
            else:
                print(f"WARNING: {col} values differ between CHIRPS and ERA5-Land datasets")

    # Sort by city and date
    merged_df['date'] = pd.to_datetime(merged_df['date'], format='%Y-%m')
    merged_df = merged_df.sort_values(['city', 'date'])
    merged_df['date'] = merged_df['date'].dt.strftime('%Y-%m')

    # --- 6. Save to CSV ---
    print("\nSaving merged data to CSV...")
    merged_df.to_csv(output_csv_path, index=False)
    print(f"Successfully created CSV file at: {output_csv_path}")
    print(f"Final dataset has {merged_df.shape[0]} rows and {merged_df.shape[1]} columns.")

    # Check for columns with all NaN values
    nan_columns = merged_df.columns[merged_df.isna().all()].tolist()
    if nan_columns:
        print(f"WARNING: The following columns contain only NaN values: {nan_columns}")

# --- Main Execution Block ---
if __name__ == '__main__':
    CHIRPS_CSV = 'chirps_city_rainfall_metrics.csv'
    ERA5_CSV = 'combined_city_weather_and_static_data.csv'
    OUTPUT_CSV = 'merged_chirps_era5_city_data.csv'
    merge_chirps_era5_csv(CHIRPS_CSV, ERA5_CSV, OUTPUT_CSV)

In [ ]:
import pandas as pd
import numpy as np

# Load CSV data
csv_file = 'merged_chirps_era5_city_data.csv'  # Replace with your file path
try:
    df = pd.read_csv(csv_file, skipinitialspace=True, encoding='utf-8')
except Exception as e:
    print(f"Error loading CSV: {e}")
    raise

# Verify columns
print("Available columns:", df.columns.tolist())
required_columns = ['date', 'rainfall_mm', 'temperature_2m', 'runoff', 'total_precipitation', 'high_vegetation_cover',
                   'geopotential_height', 'soil_volume_water_content_level1', 'soil_volume_water_content_level2',
                   'soil_volume_water_content_level3', 'soil_volume_water_content_level4', 'city']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}. Available columns: {df.columns.tolist()}")

# Convert date to datetime
try:
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m', errors='coerce')
except Exception as e:
    print(f"Error converting 'date' to datetime: {e}")
    raise

# Handle invalid dates
if df['date'].isna().any():
    print(f"Warning: {df['date'].isna().sum()} invalid dates found. Filling with forward fill.")
    df['date'] = df['date'].fillna(method='ffill')

# 1. Monsoon Intensity (0-16, Integer)
min_rainfall = df['rainfall_mm'].min()
max_rainfall = df['rainfall_mm'].max()
df['monsoon_intensity'] = np.round(16 * (df['rainfall_mm'] - min_rainfall) / (max_rainfall - min_rainfall + 1e-6)).astype(int)

# 2. Climate Change (0-17, Integer)
temp_moving_avg = df.groupby('city')['temperature_2m'].transform(lambda x: x.rolling(window=3, min_periods=3).mean())
min_temp = temp_moving_avg.min()
max_temp = temp_moving_avg.max()
temp_scaled = 8.5 * (temp_moving_avg - min_temp) / (max_temp - min_temp + 1e-6)
rainfall_threshold = df['rainfall_mm'].quantile(0.75)
df['climate_change'] = 8.5 * (df['rainfall_mm'] > rainfall_threshold).astype(int)
df['climate_change'] = np.round(temp_scaled.fillna(0) + df['climate_change']).astype(int)
df['climate_change'] = df['climate_change'].clip(0, 17)

# 3. Siltation (0-16, Integer)
df['siltation'] = (df['runoff'] * df['total_precipitation']) / (df['high_vegetation_cover'] + 1e-6)
min_siltation = df['siltation'].min()
max_siltation = df['siltation'].max()
df['siltation'] = np.round(16 * (df['siltation'] - min_siltation) / (max_siltation - min_siltation + 1e-6)).astype(int)

# 4. Agricultural Practices (0-16, Integer)
df['soil_moisture_avg'] = df[['soil_volume_water_content_level1', 'soil_volume_water_content_level2',
                             'soil_volume_water_content_level3', 'soil_volume_water_content_level4']].mean(axis=1)
min_soil = df['soil_moisture_avg'].min()
max_soil = df['soil_moisture_avg'].max()
soil_scaled = (df['soil_moisture_avg'] - min_soil) / (max_soil - min_soil + 1e-6)
min_veg = df['low_vegetation_cover'].min()
max_veg = df['low_vegetation_cover'].max()
veg_scaled = (df['low_vegetation_cover'] - min_veg) / (max_veg - min_veg + 1e-6)
df['agricultural_practices'] = np.round(16 * (soil_scaled + veg_scaled) / 2).astype(int)

# 5. Landslide Risks (0-16, Integer)
df['landslide_risks'] = (df['total_precipitation'] * df['runoff'] * df['geopotential_height']) / (df['high_vegetation_cover'] + 1e-6)
min_landslide = df['landslide_risks'].min()
max_landslide = df['landslide_risks'].max()
df['landslide_risks'] = np.round(16 * (df['landslide_risks'] - min_landslide) / (max_landslide - min_landslide + 1e-6)).astype(int)

# Drop temporary column
df = df.drop(columns=['soil_moisture_avg'])

# Save the updated dataset
df.to_csv('datasets/dataset_with_targets.csv', index=False)

print("Added Exactly 5 Target Columns (Integer Values):")
print("- Monsoon Intensity (0-16): 'monsoon_intensity' (monthly rainfall_mm scaled)")
print("- Climate Change (0-17): 'climate_change' (sum of scaled 3-month temperature moving average and monthly extreme rainfall indicator)")
print("- Siltation (0-16): 'siltation' (runoff * total_precipitation / high_vegetation_cover)")
print("- Agricultural Practices (0-16): 'agricultural_practices' (average of normalized soil_moisture_avg and high_vegetation_cover)")
print("- Landslide Risks (0-16): 'landslide_risks' ((total_precipitation * runoff * geopotential_height) / high_vegetation_cover)")
print("\nUpdated dataset saved as 'dataset_with_targets.csv'")

Available columns: ['city', 'date', 'geopotential_height', 'high_vegetation_cover', 'high_vegetation_type', 'lake_cover', 'land_sea_mask', 'low_vegetation_cover', 'low_vegetation_type', 'soil_type', 'target_latitude', 'target_longitude', 'grid_latitude', 'grid_longitude', 'total_precipitation', 'runoff', 'evaporation', 'dewpoint_temperature_2m', 'experiment_version', 'skin_reservoir_content', 'skin_temperature', 'soil_temperature_level1', 'soil_temperature_level2', 'soil_temperature_level3', 'soil_temperature_level4', 'soil_volume_water_content_level1', 'soil_volume_water_content_level2', 'soil_volume_water_content_level3', 'soil_volume_water_content_level4', 'surface_pressure', 'temperature_2m', 'u_component_wind_10m', 'v_component_wind_10m', 'rainfall_mm', 'rainfall_monthly_anomaly', 'rainfall_3month_avg', 'rainfall_6month_avg']
Added Exactly 5 Target Columns (Integer Values):
- Monsoon Intensity (0-16): 'monsoon_intensity' (monthly rainfall_mm scaled)
- Climate Change (0-17): 'clima

In [ ]:
import pandas as pd
import numpy as np

def derive_target_variables(file_path, column_names):
    """
    Loads climate data, derives target variables based on available features,
    and saves the result to a new CSV file.

    Args:
        file_path (str): The path to the input CSV file.
        column_names (list): The correct list of column names for the dataset.

    Returns:
        pandas.DataFrame: A DataFrame with the new derived target variables.
    """
    try:
        # --- 1. Load Data and Correct Column Names ---
        # Load the CSV file directly from the path.
        # We pass the correct column names to ensure the dataframe is read correctly.
        df = pd.read_csv(file_path, header=0, names=column_names)

        print("Successfully loaded data. Columns verified.")

        # FIX: The date column is named 'date' as per your new list.
        date_column = 'date'
        df[date_column] = pd.to_datetime(df[date_column])
        
        # Create a 'year' column for easier grouping.
        df['year'] = df[date_column].dt.year

        # --- 2. Monsoon Intensity (Scale: 0-16) ---
        # Define monsoon months (March-May, October-November)
        df['monsoon_month'] = df[date_column].apply(lambda x: 1 if x.month in [3, 4, 5, 10, 11] else 0)
        
        # Use 'rainfall_mm' as in the original script.
        monsoon_rainfall = df[df['monsoon_month'] == 1].groupby(['city', 'year'])['rainfall_mm'].mean().reset_index()
        monsoon_rainfall.columns = ['city', 'year', 'monsoon_intensity']
        
        # Scale to 0-16
        min_monsoon = monsoon_rainfall['monsoon_intensity'].min()
        max_monsoon = monsoon_rainfall['monsoon_intensity'].max()
        monsoon_rainfall['monsoon_intensity'] = 16 * (monsoon_rainfall['monsoon_intensity'] - min_monsoon) / (max_monsoon - min_monsoon + 1e-9)
        
        # Merge back to main dataframe
        df = df.merge(monsoon_rainfall[['city', 'year', 'monsoon_intensity']], on=['city', 'year'], how='left')

        # --- 3. Climate Change Indicators (Scale: 0-17) ---
        # a. 60-day moving average of temperature (scale to 0-8.5)
        # Use 'temperature_2m' as per the column list.
        df['temperature_moving_avg'] = df.groupby('city')['temperature_2m'].transform(
            lambda x: x.rolling(window=60, min_periods=60).mean()
        )
        min_temp = df['temperature_moving_avg'].min()
        max_temp = df['temperature_moving_avg'].max()
        df['temp_scaled'] = 8.5 * (df['temperature_moving_avg'] - min_temp) / (max_temp - min_temp + 1e-9)
        
        # b. Extreme rainfall events (scale to 0-8.5)
        # Use 'rainfall_mm' as in the original script.
        rainfall_threshold = df['rainfall_mm'].quantile(0.9)
        df['extreme_rainfall_events'] = (df['rainfall_mm'] > rainfall_threshold).astype(int)
        
        extreme_rainfall = df.groupby(['city', 'year'])['extreme_rainfall_events'].sum().reset_index()
        extreme_rainfall.columns = ['city', 'year', 'extreme_rainfall_count']
        min_extreme = extreme_rainfall['extreme_rainfall_count'].min()
        max_extreme = extreme_rainfall['extreme_rainfall_count'].max()
        extreme_rainfall['extreme_rainfall_scaled'] = 8.5 * (extreme_rainfall['extreme_rainfall_count'] - min_extreme) / (max_extreme - min_extreme + 1e-9)
        
        # Merge and combine
        df = df.merge(extreme_rainfall[['city', 'year', 'extreme_rainfall_scaled']], on=['city', 'year'], how='left')
        df['climate_change_indicators'] = df['temp_scaled'].fillna(0) + df['extreme_rainfall_scaled'].fillna(0)
        
        # --- 4. Calculations Using Newly Confirmed Columns ---
        # These sections are now enabled because 'high_vegetation_cover' is confirmed to exist.

        # Siltation Levels (Scale: 0-16)
        # Using 'runoff' and 'total_precipitation' as per the original logic and new column list.
        df['siltation_levels'] = (df['runoff'] * df['total_precipitation']) / (df['high_vegetation_cover'] + 1e-9)
        min_siltation = df['siltation_levels'].min()
        max_siltation = df['siltation_levels'].max()
        df['siltation_levels'] = 16 * (df['siltation_levels'] - min_siltation) / (max_siltation - min_siltation + 1e-9)

        # Agricultural Practices Impact (Scale: 0-16)
        # Column names for soil moisture are correct as per the new list.
        df['soil_moisture_avg'] = df[['soil_volume_water_content_level1', 'soil_volume_water_content_level2',
                                     'soil_volume_water_content_level3', 'soil_volume_water_content_level4']].mean(axis=1)
        df['agricultural_impact'] = df['soil_moisture_avg'] * df['high_vegetation_cover']
        min_agri = df['agricultural_impact'].min()
        max_agri = df['agricultural_impact'].max()
        df['agricultural_impact'] = 16 * (df['agricultural_impact'] - min_agri) / (max_agri - min_agri + 1e-9)

        # Landslide Risks (Scale: 0-16)
        # Using 'geopotential_height' directly as it exists in the new column list.
        df['landslide_risk'] = (df['total_precipitation'] * df['runoff'] * df['geopotential_height']) / (df['high_vegetation_cover'] + 1e-9)
        min_landslide = df['landslide_risk'].min()
        max_landslide = df['landslide_risk'].max()
        df['landslide_risk'] = 16 * (df['landslide_risk'] - min_landslide) / (max_landslide - min_landslide + 1e-9)

        # --- 5. Clean Up and Save ---
        # Drop intermediate columns used for calculations
        final_df = df.drop(columns=[
            'year', 'monsoon_month', 'temperature_moving_avg', 'temp_scaled', 
            'extreme_rainfall_events', 'extreme_rainfall_scaled', 'soil_moisture_avg'
        ])

        # Save the updated dataset
        output_path = 'dataset_with_all_scaled_targets.csv'
        final_df.to_csv(output_path, index=False)

        print("\n--- Processing Complete ---")
        print("Derived and Scaled Target Variables Added:")
        print("- 'monsoon_intensity' (0-16)")
        print("- 'climate_change_indicators' (0-17)")
        print("- 'siltation_levels' (0-16)")
        print("- 'agricultural_impact' (0-16)")
        print("- 'landslide_risk' (0-16)")
        print(f"\nUpdated dataset saved as '{output_path}'")
        
        return final_df

    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except KeyError as e:
        print(f"Error: A required column was not found in the CSV: {e}. Please verify the column list.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

# --- How to use the function ---
# The correct list of column names as provided by you.
column_names = [
    "city","date","geopotential_height","high_vegetation_cover","high_vegetation_type",
    "lake_cover","land_sea_mask","low_vegetation_cover","low_vegetation_type","soil_type",
    "target_latitude","target_longitude","grid_latitude","grid_longitude","total_precipitation",
    "runoff","evaporation","dewpoint_temperature_2m","experiment_version","skin_reservoir_content",
    "skin_temperature","soil_temperature_level1","soil_temperature_level2","soil_temperature_level3",
    "soil_temperature_level4","soil_volume_water_content_level1","soil_volume_water_content_level2",
    "soil_volume_water_content_level3","soil_volume_water_content_level4","surface_pressure",
    "temperature_2m","u_component_wind_10m","v_component_wind_10m","rainfall_mm",
    "rainfall_monthly_anomaly","rainfall_3month_avg","rainfall_6month_avg"
]

csv_file_path = 'merged_chirps_era5_city_data.csv' 
updated_df = derive_target_variables(csv_file_path, column_names)

# Display a sample of the final data
if updated_df is not None:
    print("\n--- Sample of Final Data for a Single City ---")
    display_cols = [
        'date', 'city', 'monsoon_intensity', 'climate_change_indicators', 
        'siltation_levels', 'agricultural_impact', 'landslide_risk'
    ]
    # Use .iloc to handle potential empty slices gracefully
    sample_df = updated_df[updated_df['city'] == updated_df['city'].unique()[0]]
    if not sample_df.empty:
        print(sample_df[display_cols].tail(10))

